# Raspberry Pi preprocessing handoff

This notebook converts one Raspberry Pi JSON record (image, temperature, humidity, gas, timestamp) into ready-to-use model inputs and writes `prepared_input.npz`. It performs **all image resizing and range conversion** and builds both model-specific sensor vectors. `predict_all_models.py` only loads the prepared arrays and calls the models.

Routing uses the classification result: **unripe ? raw-to-ripe days**, **ripe ? RSL fusion remaining-shelf-life days**. Other classes return the classification without a days estimate.

### Raspberry Pi input file

Save one captured sample as `my files/pi_input.json` and place its image beside that file (or use an absolute image path). Example:

```json
{
  "image": "capture.jpg",
  "temperature": 25.1,
  "humidity": 57.0,
  "gas": 185.0,
  "timestamp": "2026-11-21T14:30:00+05:30"
}
```

## Prepare each model's inputs

Run this code cell for each Pi sample. It resizes the image once and writes the 0?1 tensor for the Keras models plus the 0?255 tensor for the TFLite fusion model. It also builds both model-specific sensor vectors. The prediction script will load these prepared arrays without resizing or normalizing the image.

In [9]:
from pathlib import Path
from datetime import datetime
import json
import pickle
import sys
import numpy as np
import pandas as pd
from PIL import Image

HERE = Path.cwd()
# If Jupyter started in a different directory, set PROJECT to this repository root.
PROJECT = HERE if (HERE / "ml" / "rsl_multimodal").exists() else HERE.parent
INPUT_PATH = PROJECT / "my files" / "pi_input.json"
RAW39_BUNDLE = PROJECT / "my files" / "raw_to_ripe_preprocessing.pkl"
RSL_ROOT = PROJECT / "ml" / "rsl_multimodal"
PREPARED_PATH = PROJECT / "my files" / "prepared_input.npz"
RSL_STATE_PATH = PROJECT / "my files" / "rsl_sensor_state.json"

payload = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
image_path = Path(payload["image"])
if not image_path.is_absolute():
    image_path = (INPUT_PATH.parent / image_path).resolve()
if not image_path.is_file():
    raise FileNotFoundError(f"Image not found: {image_path}")
# Shared pixel geometry, separate value ranges per model contract.
with Image.open(image_path) as source_image:
    resized_image = source_image.convert("RGB").resize((224, 224), Image.Resampling.LANCZOS)
    image01 = np.asarray(resized_image, dtype=np.float32) / 255.0
image255 = image01 * 255.0
if image01.shape != (224, 224, 3) or not np.isfinite(image01).all():
    raise ValueError(f"Unexpected preprocessed image: {image01.shape}")
readings = {"Temperature": float(payload["temperature"]),
            "Humidity": float(payload["humidity"]), "Gas": float(payload["gas"])}
if not np.isfinite(list(readings.values())).all():
    raise ValueError("Temperature, humidity, and gas readings must be finite")

# RAW_TO_RIPE_REGRESSION 39-feature path. This mirrors create_live_sensor_features.
if not RAW39_BUNDLE.is_file():
    raise FileNotFoundError(
        f"Missing {RAW39_BUNDLE}. Run the export cell at the end of "
        "RAW_TO_RIPE_REGRESSION (1).ipynb in Colab, then copy the bundle here."
    )
with RAW39_BUNDLE.open("rb") as f:
    raw39 = pickle.load(f)
context = raw39["context"]
scaler39 = raw39["sensor_scaler"]
encoder39 = raw39["categorical_encoder"]
numeric_columns = list(raw39["numeric_columns"])
for required in ("Temperature_excel", "Humidity_excel", "Gas_excel", "Humidity_Level", "Gas_Level"):
    if required not in context.columns:
        raise ValueError(f"39-feature context is missing column {required}")
stds = [max(float(context[f"{name}_excel"].std()), 1e-6)
        for name in ("Temperature", "Humidity", "Gas")]
dist = sum(((context[f"{name}_excel"] - readings[name]) / std) ** 2
           for name, std in zip(("Temperature", "Humidity", "Gas"), stds))
row = context.loc[dist.idxmin()].copy()
for name in readings:
    row[f"{name}_excel"] = readings[name]
numeric_df = pd.DataFrame([row[numeric_columns].values], columns=numeric_columns)
for name in readings:
    col = f"{name}_excel"
    if col in numeric_df:
        numeric_df.loc[0, col] = readings[name]
numeric_scaled = scaler39.transform(numeric_df).astype(np.float32)
cat_columns = list(getattr(encoder39, "feature_names_in_", ["Humidity_Level", "Gas_Level"]))
cat_df = pd.DataFrame([{c: row[c] for c in cat_columns}], columns=cat_columns)
cat_encoded = encoder39.transform(cat_df)
if hasattr(cat_encoded, "toarray"):
    cat_encoded = cat_encoded.toarray()
sensor39 = np.concatenate([numeric_scaled, np.asarray(cat_encoded, dtype=np.float32)], axis=1)
if sensor39.shape != (1, 39):
    raise ValueError(f"Raw-to-ripe preprocessing made {sensor39.shape}; expected (1, 39). Check exported training artifacts.")

# RSL fusion 27-feature path. This matches scripts/predict_pi.py.
sys.path.insert(0, str(RSL_ROOT / "scripts"))
from pi_features import SensorFeatureBuilder
config = json.loads((RSL_ROOT / "artifacts" / "preprocessing_config.json").read_text())
with (RSL_ROOT / "artifacts" / "sensor_scaler.pkl").open("rb") as f:
    rsl_bundle = pickle.load(f)
feature_names = rsl_bundle["features"]["full"]
scaler27 = rsl_bundle["scalers"]["blocked__full"]
state = json.loads(RSL_STATE_PATH.read_text()) if RSL_STATE_PATH.exists() else None
builder = SensorFeatureBuilder.from_state(state, config["outlier_bounds"])
now = datetime.fromisoformat(payload["timestamp"].replace("Z", "+00:00"))
prev = None
if state and state.get("last_timestamp"):
    prev = datetime.fromisoformat(state["last_timestamp"])
minutes = (now - prev).total_seconds() / 60 if prev else 2.0
if minutes <= 0:
    raise ValueError("timestamp must advance for each sample in a batch")
raw27 = builder.vector(readings, minutes / 60.0, feature_names)
sensor27 = scaler27.transform(raw27.astype(np.float64)).astype(np.float32)
state_out = builder.state_dict()
state_out["last_timestamp"] = now.isoformat()
RSL_STATE_PATH.write_text(json.dumps(state_out, indent=2), encoding="utf-8")

np.savez_compressed(
    PREPARED_PATH,
    image_01=image01[None, ...],
    image_255=image255[None, ...],
    sensor39=sensor39.astype(np.float32),
    sensor27=sensor27.astype(np.float32),
    class_names=np.asarray(["unripe", "ripe", "overripe", "rotten"]),
    image_path=np.asarray(str(image_path)),
)
print("Prepared model inputs:", PREPARED_PATH)
print("Image tensors:", image01.shape, image01.dtype, image255.min(), image255.max())
print("Sensor vectors:", sensor39.shape, sensor27.shape)

Prepared model inputs: c:\Users\Admin\Downloads\Shelf-Life-Prediction-Of-Food-Item-using-Edge-AI\my files\prepared_input.npz
Image tensors: (224, 224, 3) float32 0.0 255.0
Sensor vectors: (1, 39) (1, 27)


c:\Users\Admin\Downloads\Shelf-Life-Prediction-Of-Food-Item-using-Edge-AI\.venv-model\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
